# 🏆 مشروع توقع كأس العالم 2026
## المرحلة الأولى: جمع البيانات وتنظيفها

| الداتاسيت | المصدر | المحتوى |
|-----------|--------|---------|
| **DS-1** | martj42 (GitHub) | 49K+ مباراة دولية 1872–2026 + ركلات ترجيح |
| **DS-2** | Kaggle | ELO ratings للـ 48 فريق (تنزيل يدوي) |
| **DS-3** | Kaggle | FIFA World Rankings 1992–2024 (تنزيل يدوي) |
| **DS-4** | Kaggle | WC 2026 Live — لاعبين + نتائج (تنزيل يدوي) |
| **DS-5** | Kaggle | EAFC 26 تقييمات اللاعبين (تنزيل يدوي) |

> **تشغيل:** نفّذ الـ cells بالترتيب من الأول للآخر


## 📦 1. استيراد المكتبات وإعداد المجلدات

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# ─── مجلد الحفظ ───────────────────────────────────────────
OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("✅ المكتبات جاهزة — المجلدات اتعملت")
print(f"   data/raw/        ← الداتا الخام")
print(f"   data/processed/  ← الداتا المنظّفة")


✅ المكتبات جاهزة — المجلدات اتعملت
   data/raw/        ← الداتا الخام
   data/processed/  ← الداتا المنظّفة


## 📥 2. تحميل DS-1 — نتائج المباريات الدولية (martj42)

**المصدر:** GitHub — بيتحدث يومياً مع مباريات كأس العالم الجارية  
**الملفات:** `results.csv` + `shootouts.csv`


In [2]:
BASE_MJ = "https://raw.githubusercontent.com/martj42/international_results/master/"

results   = pd.read_csv(BASE_MJ + "results.csv",   parse_dates=["date"])
shootouts = pd.read_csv(BASE_MJ + "shootouts.csv", parse_dates=["date"])

# حفظ الملفات الخام
results.to_csv(f"{OUTPUT_DIR}/results_raw.csv",     index=False)
shootouts.to_csv(f"{OUTPUT_DIR}/shootouts_raw.csv", index=False)

print(f"✅ DS-1 محمّل بنجاح")
print(f"   results.csv     : {len(results):,} مباراة")
print(f"   shootouts.csv   : {len(shootouts):,} مباراة ركلات ترجيح")
print(f"   آخر مباراة      : {results['date'].max().date()}")
print(f"   مباريات WC 2026 : {len(results[results['tournament']=='FIFA World Cup'])}")


✅ DS-1 محمّل بنجاح
   results.csv     : 49,477 مباراة
   shootouts.csv   : 678 مباراة ركلات ترجيح
   آخر مباراة      : 2026-06-27
   مباريات WC 2026 : 1036


### 👀 معاينة البيانات

In [3]:
print("=== أول 5 صفوف من results.csv ===")
display(results.head())

print("\n=== توزيع البطولات (أعلى 10) ===")
display(results['tournament'].value_counts().head(10).to_frame())


=== أول 5 صفوف من results.csv ===


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False



=== توزيع البطولات (أعلى 10) ===


,count
tournament,
Friendly,18388
FIFA World Cup qualification,8771
UEFA Euro qualification,2824
African Cup of Nations qualification,2327
FIFA World Cup,1036
Copa América,869
African Cup of Nations,845
AFC Asian Cup qualification,829
UEFA Nations League,658


## 🔧 3. توحيد أسماء الفرق

> أهم خطوة في المشروع — نفس الفريق ممكن يظهر بأسماء مختلفة عبر الداتاسيتات  
> مثال: `"IR Iran"` و `"Iran"` و `"IRI"` كلهم نفس المنتخب


In [4]:
TEAM_NAME_MAP = {
    "IR Iran"                    : "Iran",
    "Korea Republic"             : "South Korea",
    "Korea DPR"                  : "North Korea",
    "Côte d'Ivoire"              : "Ivory Coast",
    "Cote d'Ivoire"              : "Ivory Coast",
    "USA"                        : "United States",
    "Trinidad & Tobago"          : "Trinidad and Tobago",
    "Bosnia-Herzegovina"         : "Bosnia and Herzegovina",
    "Bosnia & Herzegovina"       : "Bosnia and Herzegovina",
    "Czechia"                    : "Czech Republic",
    "Czech Rep."                 : "Czech Republic",
    "Congo DR"                   : "DR Congo",
    "Dem. Rep. Congo"            : "DR Congo",
    "Chinese Taipei"             : "Taiwan",
    "Cabo Verde"                 : "Cape Verde",
    "UAE"                        : "United Arab Emirates",
    "Kyrgyz Republic"            : "Kyrgyzstan",
    "FYR Macedonia"              : "North Macedonia",
    "Slovak Republic"            : "Slovakia",
    "St. Kitts & Nevis"          : "Saint Kitts and Nevis",
    "St. Vincent & Grenadines"   : "Saint Vincent and the Grenadines",
    "São Tomé & Príncipe"        : "Sao Tome and Principe",
    "Eswatini"                   : "Swaziland",
    "Timor-Leste"                : "East Timor",
}

def normalize_team(name):
    """توحيد اسم الفريق عبر جميع الداتاسيتات"""
    if pd.isna(name):
        return name
    return TEAM_NAME_MAP.get(str(name).strip(), str(name).strip())

# تطبيق التوحيد
for col in ["home_team", "away_team"]:
    results[col]   = results[col].apply(normalize_team)
    shootouts[col] = shootouts[col].apply(normalize_team)
shootouts["winner"] = shootouts["winner"].apply(normalize_team)

print(f"✅ تم توحيد {len(TEAM_NAME_MAP)} اسم فريق مختلف")


✅ تم توحيد 24 اسم فريق مختلف


## 🧹 4. تنظيف النتائج وإضافة أعمدة مهمة

- فصل المباريات المنتهية عن القادمة  
- إضافة: النتيجة + فارق الأهداف + وزن أهمية البطولة


In [5]:
# فصل المباريات المنتهية عن القادمة
results_played = results.dropna(subset=["home_score", "away_score"]).copy()
results_future = results[results["home_score"].isna()].copy()

print(f"مباريات منتهية : {len(results_played):,}")
print(f"مباريات قادمة  : {len(results_future):,}")

# ── نتيجة المباراة ──────────────────────────────────────
def get_result(row):
    if row["home_score"] > row["away_score"]: return "home_win"
    elif row["home_score"] < row["away_score"]: return "away_win"
    else: return "draw"

results_played["result"]      = results_played.apply(get_result, axis=1)
results_played["goal_diff"]   = results_played["home_score"] - results_played["away_score"]
results_played["total_goals"] = results_played["home_score"] + results_played["away_score"]

# ── وزن أهمية البطولة ───────────────────────────────────
TOURNAMENT_WEIGHTS = {
    "FIFA World Cup"               : 4,
    "FIFA World Cup qualification" : 2,
    "Copa América"                 : 3,
    "UEFA Euro"                    : 3,
    "UEFA Euro qualification"      : 2,
    "African Cup of Nations"       : 3,
    "AFC Asian Cup"                : 3,
    "CONCACAF Gold Cup"            : 3,
    "UEFA Nations League"          : 2,
    "Friendly"                     : 1,
}
results_played["tournament_weight"] = results_played["tournament"].map(
    lambda t: TOURNAMENT_WEIGHTS.get(t, 1)
)

# المباريات الرسمية فقط (وزن > 1)
results_official = results_played[results_played["tournament_weight"] > 1].copy()

print(f"\n✅ مباريات رسمية  : {len(results_official):,}")
print(f"   مباريات ودية   : {len(results_played[results_played['tournament_weight']==1]):,}")
print("\n--- توزيع النتائج ---")
display(results_played["result"].value_counts().to_frame())


مباريات منتهية : 49,459
مباريات قادمة  : 18

✅ مباريات رسمية  : 15,794
   مباريات ودية   : 33,665

--- توزيع النتائج ---


,count
result,
home_win,24241
away_win,13971
draw,11247


## ⚡ 5. حساب ELO Ratings

> ELO نظام تصنيف ديناميكي أدق من FIFA Ranking — بيتحدث بعد كل مباراة  
> بنستخدم نسخة معدّلة تأخذ في الاعتبار:
> - **وزن البطولة** — WC أهم من الودية
> - **فارق الأهداف** — فوز 4-0 أثقل من 1-0
> - **ميزة الأرض** — +100 نقطة للمضيف


In [6]:
def calculate_elo(results_df, k_base=32, initial_rating=1500):
    """
    حساب ELO rating لكل فريق عبر التاريخ.
    المدخلات: DataFrame بالمباريات مرتبة تصاعدياً بالتاريخ
    المخرجات: (dict ratings, DataFrame history)
    """
    ratings = {}

    def get_rating(team):
        return ratings.get(team, initial_rating)

    def expected_score(ra, rb):
        return 1 / (1 + 10 ** ((rb - ra) / 400))

    def goal_index(gd):
        """معامل تأثير فارق الأهداف على K"""
        if abs(gd) <= 1:   return 1.0
        elif abs(gd) == 2: return 1.5
        else:              return (11 + abs(gd)) / 8

    history = []

    for _, row in results_df.sort_values("date").iterrows():
        home, away = row["home_team"], row["away_team"]
        hs, as_ = row["home_score"], row["away_score"]
        weight  = row.get("tournament_weight", 1)

        ra, rb = get_rating(home), get_rating(away)

        w_home = 1.0 if hs > as_ else (0.0 if hs < as_ else 0.5)
        w_away = 1.0 - w_home

        home_adv = 0 if row.get("neutral", False) else 100
        e_home   = expected_score(ra + home_adv, rb)
        e_away   = 1 - e_home

        K        = k_base * weight * goal_index(abs(hs - as_))
        new_ra   = ra + K * (w_home - e_home)
        new_rb   = rb + K * (w_away - e_away)

        ratings[home], ratings[away] = new_ra, new_rb

        history.append({
            "date"           : row["date"],
            "home_team"      : home,
            "away_team"      : away,
            "elo_home_before": round(ra, 1),
            "elo_away_before": round(rb, 1),
            "elo_home_after" : round(new_ra, 1),
            "elo_away_after" : round(new_rb, 1),
        })

    return ratings, pd.DataFrame(history)

# ── تشغيل ELO على المباريات الرسمية فقط ───────────────
elo_ratings, elo_history_df = calculate_elo(results_official)

elo_current = (
    pd.DataFrame.from_dict(elo_ratings, orient="index", columns=["elo"])
    .reset_index().rename(columns={"index": "team"})
    .sort_values("elo", ascending=False).reset_index(drop=True)
)
elo_current["rank"] = elo_current.index + 1
elo_current["elo"]  = elo_current["elo"].round(1)

# حفظ
elo_history_df.to_csv("data/processed/elo_history.csv", index=False)
elo_current.to_csv("data/processed/elo_current.csv",   index=False)

print(f"✅ ELO محسوب لـ {len(elo_current)} فريق")
print("\n🏆 أعلى 15 فريق في ELO:")
display(elo_current.head(15))


✅ ELO محسوب لـ 217 فريق

🏆 أعلى 15 فريق في ELO:


,team,elo,rank
0,Spain,2219.8,1
1,Argentina,2207.0,2
2,France,2173.2,3
3,England,2130.5,4
4,Colombia,2100.8,5
5,Norway,2081.2,6
6,Brazil,2080.1,7
7,Morocco,2039.5,8
8,Netherlands,2037.2,9
9,Germany,2032.2,10


## 🎯 6. إحصاءات ركلات الترجيح

> **الميزة المخفية:** ركلات الترجيح مقياس مباشر للـ "روح القتالية تحت الضغط"  
> موجودة في `shootouts.csv` ومحدش بيستخدمها في نماذج التوقع!


In [7]:
def build_shootout_stats(shootouts_df):
    """نسبة الفوز بركلات الترجيح لكل فريق تاريخياً"""
    stats = {}
    for _, row in shootouts_df.iterrows():
        for team in [row["home_team"], row["away_team"]]:
            if team not in stats:
                stats[team] = {"played": 0, "won": 0}
            stats[team]["played"] += 1
            if team == row["winner"]:
                stats[team]["won"] += 1

    df = pd.DataFrame.from_dict(stats, orient="index").reset_index()
    df.columns = ["team", "shootout_played", "shootout_won"]
    df["shootout_win_rate"] = (df["shootout_won"] / df["shootout_played"]).round(3)
    return df.sort_values("shootout_played", ascending=False).reset_index(drop=True)

shootout_stats = build_shootout_stats(shootouts)
shootout_stats.to_csv("data/processed/shootout_stats.csv", index=False)

print(f"✅ إحصاءات ركلات الترجيح لـ {len(shootout_stats)} فريق")
print("\n🎯 أكثر الفرق خبرة بركلات الترجيح:")
display(shootout_stats.head(12))


✅ إحصاءات ركلات الترجيح لـ 230 فريق

🎯 أكثر الفرق خبرة بركلات الترجيح:


,team,shootout_played,shootout_won,shootout_win_rate
0,South Africa,28,13,0.464
1,Egypt,26,15,0.577
2,Zambia,25,14,0.560
3,South Korea,25,15,0.600
4,Iran,23,10,0.435
5,Argentina,23,15,0.652
6,Thailand,22,13,0.591
7,Senegal,21,11,0.524
8,Cameroon,18,9,0.500
9,Malawi,18,4,0.222


## 📈 7. دالة حساب الفورم الحديثة

> بنستخدمها في المرحلة الثانية لحساب فورم كل فريق قبل كل مباراة


In [8]:
def calculate_recent_form(results_df, team, before_date=None, n_matches=10, official_only=True):
    """
    حساب فورم الفريق في آخر N مباريات قبل تاريخ معين.
    
    المعاملات:
        results_df   : DataFrame النتائج المنظّفة
        team         : اسم الفريق
        before_date  : تاريخ المباراة (نحسب الفورم قبله فقط — مهم جداً لتجنب data leakage)
        n_matches    : عدد المباريات اللي بنحسب عليها الفورم
        official_only: رسمية فقط أم كل المباريات
    
    يرجع: dict بإحصاءات الفورم
    """
    df = results_df[results_df["tournament_weight"] > 1].copy() if official_only else results_df.copy()
    
    if before_date:
        df = df[df["date"] < pd.Timestamp(before_date)]

    home_g = df[df["home_team"] == team].copy()
    home_g["scored"]   = home_g["home_score"]
    home_g["conceded"] = home_g["away_score"]
    home_g["points"]   = home_g["result"].map({"home_win":3, "draw":1, "away_win":0})

    away_g = df[df["away_team"] == team].copy()
    away_g["scored"]   = away_g["away_score"]
    away_g["conceded"] = away_g["home_score"]
    away_g["points"]   = away_g["result"].map({"away_win":3, "draw":1, "home_win":0})

    all_g = pd.concat([home_g, away_g]).sort_values("date", ascending=False).head(n_matches)

    if len(all_g) == 0:
        return {"form_points":0, "form_gf":0, "form_ga":0, "form_gd":0,
                "form_win_rate":0, "form_matches":0}

    return {
        "form_points"  : int(all_g["points"].sum()),
        "form_gf"      : round(all_g["scored"].mean(), 2),
        "form_ga"      : round(all_g["conceded"].mean(), 2),
        "form_gd"      : round((all_g["scored"] - all_g["conceded"]).mean(), 2),
        "form_win_rate": round((all_g["points"] == 3).mean(), 3),
        "form_matches" : len(all_g),
    }

# ── اختبار سريع ────────────────────────────────────────
teams_to_test = ["Argentina", "France", "Brazil", "Spain", "England", "Morocco"]
print("🔥 فورم أقوى الفرق (آخر 10 مباريات رسمية):\n")
form_data = []
for t in teams_to_test:
    f = calculate_recent_form(results_played, t)
    form_data.append({"الفريق": t, **f})
display(pd.DataFrame(form_data))


🔥 فورم أقوى الفرق (آخر 10 مباريات رسمية):



,الفريق,form_points,form_gf,form_ga,form_gd,form_win_rate,form_matches
0,Argentina,22,1.7,0.5,1.2,0.7,10
1,France,25,2.8,1.0,1.8,0.8,10
2,Brazil,18,1.5,0.8,0.7,0.5,10
3,Spain,24,3.2,0.8,2.4,0.7,10
4,England,28,2.6,0.2,2.4,0.9,10
5,Morocco,24,1.8,0.4,1.4,0.7,10


## 💾 8. حفظ جميع البيانات المنظّفة

In [9]:
results_played.to_csv("data/processed/results_clean.csv",    index=False)
results_official.to_csv("data/processed/results_official.csv", index=False)
results_future.to_csv("data/processed/results_future.csv",   index=False)
shootouts.to_csv("data/processed/shootouts_clean.csv",       index=False)

print("✅ تم الحفظ!")
print("\nالملفات الجاهزة في data/processed/:")
print("-" * 50)
for f in sorted(os.listdir("data/processed")):
    size = os.path.getsize(f"data/processed/{f}") // 1024
    print(f"  {f:40s}  {size:>5} KB")


✅ تم الحفظ!

الملفات الجاهزة في data/processed/:
--------------------------------------------------
  elo_current.csv                               4 KB
  elo_history.csv                             895 KB
  results_clean.csv                          4765 KB
  results_future.csv                            1 KB
  results_official.csv                       1619 KB
  shootout_stats.csv                            4 KB
  shootouts_clean.csv                          28 KB


## ⚠️ 9. الداتاسيتات اللي محتاج تنزّلها من Kaggle

نزّلها مرة واحدة وضعها في مجلد `data/raw/` — وبعدين شغّل الكود تاني وهيكمل أوتوماتيك.

| # | الداتاسيت | اسم الملف | الرابط |
|---|-----------|-----------|--------|
| DS-2 | ELO ratings للـ 48 فريق | `elo_ratings_wc2026.csv` | [Kaggle](https://www.kaggle.com/datasets/afonsofernandescruz/2026-fifa-world-cup-historical-elo-ratings) |
| DS-3 | FIFA World Rankings | `fifa_ranking.csv` | [Kaggle](https://www.kaggle.com/datasets/cashncarry/fifaworldranking) |
| DS-4 | WC 2026 Live | `squads_and_players.csv` | [Kaggle](https://www.kaggle.com/datasets/mominullptr/fifa-world-cup-2026-dataset) |
| DS-5 | EAFC 26 Players | `eafc26_players.csv` | [Kaggle](https://www.kaggle.com/datasets/flynn28/eafc26-player-database) |

---
### ✅ المرحلة الأولى منتهية — جاهزين للمرحلة الثانية: Feature Engineering!


In [11]:
squads = pd.read_csv("data/raw/squads_and_players.csv")
print(squads.columns.tolist())
print(squads.head(3))

['player_id', 'team_id', 'player_name', 'position', 'club_team', 'market_value_eur', 'caps', 'date_of_birth', 'height_cm', 'goals']
   player_id  team_id            player_name position            club_team  \
0          1        1       José Raúl Rangel       GK       CD Guadalajara   
1          2        1  Jorge Eduardo Sanchez      DEF        PAOK Saloniki   
2          3        1     César Jasib Montes      DEF  FC Lokomotiv Moscow   

   market_value_eur  caps date_of_birth  height_cm  goals  
0           6500000    15    2000-02-25        190      0  
1           2000000    59    1997-12-10        176      3  
2           7500000    69    1997-02-24        191      4  


In [12]:
# نشوف أعمدة teams.csv الأول
teams = pd.read_csv("data/raw/teams.csv")
print(teams.columns.tolist())
print(teams.head(3))

['team_id', 'team_name', 'fifa_code', 'group_letter', 'confederation', 'fifa_ranking_pre_tournament', 'elo_rating', 'manager_name']
   team_id     team_name fifa_code group_letter confederation  \
0        1        Mexico       MEX            A      CONCACAF   
1        2  South Africa       RSA            A           CAF   
2        3   South Korea       KOR            A           AFC   

   fifa_ranking_pre_tournament  elo_rating    manager_name  
0                           14        1810  Javier Aguirre  
1                           60        1620      Hugo Broos  
2                           22        1800   Hong Myung-bo  


In [13]:
# تحميل teams وsquads مع بعض
teams = pd.read_csv("data/raw/teams.csv")
squads = pd.read_csv("data/raw/squads_and_players.csv")

# merge عشان نجيب team_name من teams
squads = squads.merge(teams[["team_id", "team_name"]], on="team_id", how="left")
squads["team_name"] = squads["team_name"].apply(normalize_team)

# ملخص لكل فريق
squad_summary = squads.groupby("team_name").agg(
    avg_market_value   = ("market_value_eur", "mean"),
    total_market_value = ("market_value_eur", "sum"),
    avg_caps           = ("caps", "mean"),
    squad_size         = ("player_name", "count"),
).reset_index().rename(columns={"team_name": "team"})

squad_summary.to_csv("data/processed/squad_summary.csv", index=False)
print(f"✅ DS-4 محمّل: {len(squads):,} لاعب في {squads['team_name'].nunique()} فريق")
display(squad_summary.sort_values("total_market_value", ascending=False).head(10))

✅ DS-4 محمّل: 1,248 لاعب في 48 فريق


,team,avg_market_value,total_market_value,avg_caps,squad_size
17,France,5.857692e+07,1523000000,31.384615,26
16,England,5.223846e+07,1358200000,32.807692,26
40,Spain,4.703077e+07,1222800000,29.153846,26
33,Portugal,3.867308e+07,1005500000,45.230769,26
18,Germany,3.642308e+07,947000000,33.076923,26
6,Brazil,3.570000e+07,928200000,35.846154,26
1,Argentina,3.105769e+07,807500000,48.730769,26
28,Netherlands,2.900769e+07,754200000,33.730769,26
30,Norway,2.268846e+07,589900000,31.423077,26
4,Belgium,2.105769e+07,547500000,42.346154,26


In [16]:
# ── تحميل DS-3: FIFA Rankings ───────────────────────────
FIFA_RANK_PATH = f"{OUTPUT_DIR}/fifa_ranking.csv"
if os.path.exists(FIFA_RANK_PATH):
    fifa_rank = pd.read_csv(FIFA_RANK_PATH, parse_dates=["rank_date"])
    fifa_rank["country_full"] = fifa_rank["country_full"].apply(normalize_team)
    fifa_rank_latest = (
        fifa_rank.sort_values("rank_date").groupby("country_full").last().reset_index()
        [["country_full","rank","total_points"]]
        .rename(columns={"country_full":"team","rank":"fifa_rank","total_points":"fifa_points"})
    )
    fifa_rank_latest.to_csv("data/processed/fifa_rank_latest.csv", index=False)
    print(f"✅ DS-3 محمّل: {len(fifa_rank_latest)} فريق")
    display(fifa_rank_latest.head(10))
else:
    print("⚠️  DS-3 غير موجود — نزّله من Kaggle وضعه في data/raw/fifa_ranking.csv")

# ── تحميل DS-4: WC 2026 Live ────────────────────────────
SQUADS_PATH = f"{OUTPUT_DIR}/squads_and_players.csv"
if os.path.exists(SQUADS_PATH):
    teams_df = pd.read_csv("data/raw/teams.csv")
    squads = pd.read_csv(SQUADS_PATH)
    squads = squads.merge(teams_df[["team_id","team_name"]], on="team_id", how="left")
    squads["team_name"] = squads["team_name"].apply(normalize_team)
    squad_summary = squads.groupby("team_name").agg(
        avg_market_value   = ("market_value_eur", "mean"),
        total_market_value = ("market_value_eur", "sum"),
        avg_caps           = ("caps", "mean"),
        squad_size         = ("player_name", "count"),
    ).reset_index().rename(columns={"team_name":"team"})
    squad_summary.to_csv("data/processed/squad_summary.csv", index=False)
    print(f"\n✅ DS-4 محمّل: {len(squads):,} لاعب في {squads['team_name'].nunique()} فريق")
    display(squad_summary.sort_values("total_market_value", ascending=False).head(10))
else:
    print("⚠️  DS-4 غير موجود — نزّله من Kaggle وضعه في data/raw/squads_and_players.csv")

# ── تحميل DS-5: EAFC 26 ─────────────────────────────────
try:
    eafc = pd.read_csv("data/raw/EAFC26-Men.csv")
    print("\n✅ EAFC26-Men محمّل")
except:
    eafc = pd.read_csv("data/raw/EAFC26.csv")
    print("\n✅ EAFC26 محمّل")

eafc["Nation"] = eafc["Nation"].apply(normalize_team)

team_ratings = (
    eafc.sort_values("OVR", ascending=False)
    .groupby("Nation")
    .head(15)
    .groupby("Nation")
    .agg(
        avg_overall = ("OVR", "mean"),
        max_overall = ("OVR", "max"),
        squad_depth = ("OVR", "count"),
    )
    .reset_index()
    .rename(columns={"Nation": "team"})
)
team_ratings["avg_overall"] = team_ratings["avg_overall"].round(1)
team_ratings.to_csv("data/processed/team_ratings_fc26.csv", index=False)
print(f"✅ DS-5 محمّل: تقييم {len(team_ratings)} منتخب")
display(team_ratings.sort_values("avg_overall", ascending=False).head(10))

✅ DS-3 محمّل: 216 فريق


,team,fifa_rank,fifa_points
0,Afghanistan,151.0,1034.37
1,Albania,66.0,1379.40
2,Algeria,44.0,1474.13
3,American Samoa,188.0,890.97
4,Andorra,162.0,996.05
5,Angola,92.0,1248.94
6,Anguilla,209.0,779.86
7,Antigua and Barbuda,148.0,1052.80
8,Argentina,1.0,1860.14
9,Armenia,96.0,1230.79



✅ DS-4 محمّل: 1,248 لاعب في 48 فريق


,team,avg_market_value,total_market_value,avg_caps,squad_size
17,France,5.857692e+07,1523000000,31.384615,26
16,England,5.223846e+07,1358200000,32.807692,26
40,Spain,4.703077e+07,1222800000,29.153846,26
33,Portugal,3.867308e+07,1005500000,45.230769,26
18,Germany,3.642308e+07,947000000,33.076923,26
6,Brazil,3.570000e+07,928200000,35.846154,26
1,Argentina,3.105769e+07,807500000,48.730769,26
28,Netherlands,2.900769e+07,754200000,33.730769,26
30,Norway,2.268846e+07,589900000,31.423077,26
4,Belgium,2.105769e+07,547500000,42.346154,26



✅ EAFC26-Men محمّل
✅ DS-5 محمّل: تقييم 156 منتخب


,team,avg_overall,max_overall,squad_depth
50,France,86.1,91,15
129,Spain,85.8,90,15
19,Brazil,85.3,89,15
45,England,85.2,90,15
54,Germany,84.8,89,15
114,Portugal,84.5,89,15
74,Italy,84.4,89,15
6,Argentina,84.1,88,15
64,Holland,83.9,90,15
14,Belgium,82.7,89,15
